# Inscrição e circunscrição: corpos redondos

**Módulo 06 — Geometria Espacial** · Notebook 16 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0616a_inscricao_e_circunscricao_corpos_redondos.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0610a_piramide.ipynb`](0610a_piramide.ipynb) (relações métricas da pirâmide regular), [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb), [`0612a_cone.ipynb`](0612a_cone.ipynb) e [`0613a_esfera.ipynb`](0613a_esfera.ipynb) (os três corpos redondos), [`0614a_solidos_semelhantes_e_troncos.ipynb`](0614a_solidos_semelhantes_e_troncos.ipynb) (tronco de cone) e [`0615a_inscricao_e_circunscricao_esfera_e_poliedros.ipynb`](0615a_inscricao_e_circunscricao_esfera_e_poliedros.ipynb) (esferas inscrita e circunscrita)
- Módulo 04 — Geometria Plana: [`0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb`](../04_geometria_plana/0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb) (semelhança de triângulos), [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb) (incentro e circuncentro: circunferências inscrita e circunscrita), [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (área do triângulo, $S = p \cdot r$), [`0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb`](../04_geometria_plana/0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb) (trapézio) e [`0412a_circunferencia_e_circulo.ipynb`](../04_geometria_plana/0412a_circunferencia_e_circulo.ipynb) (quadrilátero circunscritível)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- usar a seção meridiana para transformar um problema espacial num problema plano;
- relacionar prismas e cilindros, pirâmides e cones, cilindros e cones inscritos e circunscritos;
- calcular os raios de esferas inscritas e circunscritas a cones e troncos de cone;
- reconhecer a proporção 4 : 6 : 9 entre esfera, cilindro equilátero e cone equilátero circunscritos.

**Tempo estimado:** 80 a 100 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math
import numpy as np

## Onde para a cereja?

Uma taça de coquetel tem a forma de um cone: a "boca" é larga e o fundo termina numa ponta. Você deixa cair uma cereja (uma bolinha) dentro dela. A cereja escorrega até ficar presa, encostando nas paredes da taça.

A que altura fica o centro da cereja? E ela fica totalmente dentro da taça, ou fica com um pedaço para fora?

Pensar no problema em 3D é difícil. Mas olhe a taça **de lado**, cortando-a ao meio por um plano que passa pelo eixo: a taça vira um **triângulo** (um "V") e a cereja vira um **círculo** encostado nos dois lados do V. Pronto: virou um problema de Geometria Plana!

🕹️ Mexa no tamanho da cereja e na abertura da taça.

In [2]:
# Cores usadas em todo o notebook: o sólido "de fora" e o "de dentro" têm a mesma cor
# na figura 3D e na seção meridiana (a figura 2D logo abaixo).
COR_FORA = u3d.CORES["solido"]
COR_DENTRO = u3d.CORES["plano_secundario"]
COR_TANGENCIA = u3d.CORES["destaque"]       # pontos de contato
COR_MEDIDA = u3d.CORES["reta_secundaria"]   # segmentos usados nas contas


def soltar_cereja(raio_da_cereja: float, raio_da_boca: float, altura_da_taca: float) -> None:
    """Desenha a taça e a cereja (em 3D e na seção meridiana) e calcula onde ela para."""
    geratriz = math.sqrt(altura_da_taca**2 + raio_da_boca**2)
    seno_da_semiabertura = raio_da_boca / geratriz          # ângulo entre o eixo e a parede
    # Na seção meridiana, o centro O da cereja, a ponta V da taça e o ponto T onde a
    # cereja toca a parede formam um triângulo retângulo em T (o raio OT é
    # perpendicular à parede). Nele, o cateto OT é oposto ao ângulo em V, e a
    # hipotenusa é VO: sen(semiabertura) = OT / VO.
    distancia_do_fundo_ao_centro = raio_da_cereja / seno_da_semiabertura
    topo_da_cereja = distancia_do_fundo_ao_centro + raio_da_cereja
    cabe_inteira = topo_da_cereja <= altura_da_taca

    # O ponto T fica na parede, a uma distância VT = VO · cos(semiabertura) da ponta.
    # (Se T passar da boca da taça, a cereja nem toca as paredes: fica na borda.)
    cosseno_da_semiabertura = altura_da_taca / geratriz
    VT = distancia_do_fundo_ao_centro * cosseno_da_semiabertura
    apoiada_na_borda = VT > geratriz
    if apoiada_na_borda:
        # A cereja encosta na circunferência da boca: Pitágoras no triângulo formado
        # pelo centro da cereja, o centro da boca e um ponto da borda.
        altura_do_centro = altura_da_taca + math.sqrt(raio_da_cereja**2 - raio_da_boca**2)
    else:
        altura_do_centro = distancia_do_fundo_ao_centro

    # Figura 3D: a taça é um cone de cabeça para baixo, sem a base (a boca é aberta).
    fig = u3d.criar_figura(titulo="A cereja na taça")
    fig = u3d.desenhar_cone(
        fig, (0, 0, 0), (0, 0, altura_da_taca), raio_da_boca, cor=COR_FORA,
        opacidade=0.25, mostrar_base=False,
    )
    fig = u3d.desenhar_esfera(
        fig, (0, 0, altura_do_centro), raio_da_cereja, cor=COR_TANGENCIA, opacidade=0.8
    )
    u3d.mostrar(fig)

    # Figura 2D: a seção meridiana. A taça vira um "V" e a cereja, um círculo.
    secao = u3d.criar_grafico(titulo="Seção meridiana: o V e o círculo", proporcao_igual=True)
    secao = u3d.desenhar_curva(
        secao, [-raio_da_boca, 0, raio_da_boca], [altura_da_taca, 0, altura_da_taca],
        rotulo="taça", cor=COR_FORA,
    )
    secao = u3d.desenhar_circunferencia_2d(
        secao, (0, altura_do_centro), raio_da_cereja, rotulo="cereja", cor=COR_TANGENCIA,
        preencher=True,
    )
    secao = u3d.desenhar_curva(
        secao, [-raio_da_boca, raio_da_boca], [altura_da_taca, altura_da_taca],
        cor=u3d.CORES["apoio"], espessura=1, tracejado=True,
    )
    secao = u3d.marcar_ponto_no_grafico(secao, 0, 0, rotulo="V", cor=u3d.CORES["apoio"])
    secao = u3d.marcar_ponto_no_grafico(
        secao, 0, altura_do_centro, rotulo="O", cor=u3d.CORES["apoio"]
    )
    if not apoiada_na_borda:
        # T, na parede da direita: andamos VT ao longo da direção da parede.
        T = (VT * raio_da_boca / geratriz, VT * altura_da_taca / geratriz)
        secao = u3d.desenhar_curva(
            secao, [0, 0], [0, altura_do_centro], rotulo="VO", cor=COR_MEDIDA, tracejado=True
        )
        secao = u3d.desenhar_curva(
            secao, [0, T[0]], [altura_do_centro, T[1]], rotulo="raio OT", cor=COR_MEDIDA
        )
        secao = u3d.marcar_ponto_no_grafico(secao, T[0], T[1], rotulo="T", cor=COR_TANGENCIA)
    u3d.mostrar(secao)

    if apoiada_na_borda:
        print("A cereja é grande demais para esta taça: ela nem chega a tocar as paredes e")
        print(f"fica apoiada na borda, com o centro a {altura_do_centro:.2f} cm de altura.")
        return
    print(f"Geratriz da taça: {geratriz:.2f} cm;  sen(semiabertura) = {seno_da_semiabertura:.4f}")
    print(f"Altura do centro da cereja: VO = r / sen = {distancia_do_fundo_ao_centro:.2f} cm")
    print(f"Altura do topo da cereja: {topo_da_cereja:.2f} cm "
          f"(a taça tem {altura_da_taca:g} cm)")
    if cabe_inteira:
        print("A cereja fica inteira dentro da taça.")
    else:
        print("Um pedaço da cereja fica para fora da taça.")


interact(
    soltar_cereja,
    raio_da_cereja=widgets.FloatSlider(
        min=0.5, max=2.5, step=0.25, value=1.5, description="raio da cereja (cm)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    raio_da_boca=widgets.FloatSlider(
        min=2, max=6, step=0.5, value=5, description="raio da boca (cm)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    altura_da_taca=widgets.FloatSlider(
        min=4, max=10, step=0.5, value=7, description="altura da taça (cm)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=1.5, continuous_update=False, description='raio da cereja (cm)', max=2…

Esse truque de "olhar de lado" é a ferramenta mais poderosa para os problemas de sólidos redondos inscritos uns nos outros. Neste notebook, você vai aplicá-lo a várias combinações: cilindros dentro de prismas, cones dentro de pirâmides, cilindros dentro de cones e de esferas, esferas dentro de cones e troncos de cone.

## 1. A técnica: olhar pela seção meridiana

Quando dois sólidos têm o **mesmo eixo** de simetria, um plano que contém esse eixo corta os dois ao mesmo tempo e mostra como eles se encaixam.

| Sólido | Seção meridiana |
|---|---|
| cilindro reto | retângulo $2r \times h$ |
| cone reto | triângulo isósceles de base $2r$ e altura $h$ |
| tronco de cone reto | trapézio isósceles de bases $2R$ e $2r$ e altura $h$ |
| esfera | círculo máximo de raio $R$ |

> 💡 **Dica.** Antes de fazer qualquer conta, **desenhe a seção meridiana** e marque nela os pontos de contato entre os sólidos. Os triângulos retângulos e semelhantes que aparecem nesse desenho resolvem quase tudo.

Duas ferramentas da Geometria Plana aparecem o tempo todo:
- **semelhança de triângulos** (módulo 04);
- **a circunferência inscrita num triângulo**, cujo raio é $r = \dfrac{\text{área do triângulo}}{\text{semiperímetro}}$. (Compare com o notebook 15: no espaço, era $r = \frac{3 \cdot \text{volume}}{\text{área}}$.)

Para prismas e pirâmides, que não têm seção circular, a ideia equivalente é olhar para a **base**: um cilindro inscrito num prisma tem a base inscrita na base do prisma, e assim por diante.

🕹️ Escolha um sólido e veja o plano que contém o eixo cortando-o.

In [3]:
def cortar_pelo_eixo(solido: str) -> None:
    """Desenha o sólido com o plano que contém o eixo e a seção meridiana em 2D."""
    R, r, h = 2, 1, 3   # medidas fixas, só para o desenho
    # Os vértices da seção meridiana, como pontos (x, altura) do plano do corte.
    if solido == "cilindro reto":
        secao_2d = [(-R, 0), (R, 0), (R, h), (-R, h)]
        nome_da_secao = "retângulo 2r × h"
    elif solido == "cone reto":
        secao_2d = [(-R, 0), (R, 0), (0, h)]
        nome_da_secao = "triângulo isósceles de base 2r e altura h"
    elif solido == "tronco de cone reto":
        secao_2d = [(-R, 0), (R, 0), (r, h), (-r, h)]
        nome_da_secao = "trapézio isósceles de bases 2R e 2r e altura h"
    else:
        angulos = np.linspace(0, 2 * math.pi, 60, endpoint=False)
        secao_2d = [(R * math.cos(t), R + R * math.sin(t)) for t in angulos]
        nome_da_secao = "círculo máximo de raio R"

    fig = u3d.criar_figura(titulo=f"{solido.capitalize()} cortado pelo eixo")
    if solido == "cilindro reto":
        fig = u3d.desenhar_cilindro(fig, (0, 0, 0), R, (0, 0, h), cor=COR_FORA, opacidade=0.2)
    elif solido == "cone reto":
        fig = u3d.desenhar_cone(fig, (0, 0, h), (0, 0, 0), R, cor=COR_FORA, opacidade=0.2)
    elif solido == "tronco de cone reto":
        fig = u3d.desenhar_tronco_de_cone(fig, (0, 0, 0), R, r, h, cor=COR_FORA, opacidade=0.2)
    else:
        fig = u3d.desenhar_esfera(fig, (0, 0, R), R, cor=COR_FORA, opacidade=0.2)
    # O plano do corte é vertical e contém o eixo. A seção fica pintada nele.
    fig = u3d.desenhar_plano(
        fig, (0, 0, h / 2 if solido != "esfera" else R), (0, 1, 0),
        cor=u3d.CORES["apoio"], opacidade=0.08, tamanho=6,
    )
    fig = u3d.desenhar_poligono(
        fig, [(x, 0, z) for x, z in secao_2d], cor=COR_DENTRO, opacidade=0.6
    )
    u3d.mostrar(fig)

    secao = u3d.criar_grafico(titulo=f"Seção meridiana: {nome_da_secao}", proporcao_igual=True)
    secao = u3d.desenhar_poligono_2d(secao, secao_2d, cor=COR_DENTRO)
    u3d.mostrar(secao)


interact(
    cortar_pelo_eixo,
    solido=widgets.Dropdown(
        options=["cilindro reto", "cone reto", "tronco de cone reto", "esfera"],
        value="cone reto", description="sólido",
    ),
);

interactive(children=(Dropdown(description='sólido', index=1, options=('cilindro reto', 'cone reto', 'tronco d…

## 2. Prismas, pirâmides, cilindros e cones

### Prisma e cilindro

Um **cilindro está inscrito num prisma** quando as bases do cilindro estão inscritas nas bases do prisma e os dois têm a mesma altura. Se o prisma é regular, o raio do cilindro é o **apótema da base** do prisma: $r = m$.

Um **cilindro está circunscrito a um prisma** quando as bases do prisma estão inscritas nas bases do cilindro. Se o prisma é regular, o raio do cilindro é o **raio da base** do prisma: $r = R$.

### Pirâmide e cone

Um **cone está inscrito numa pirâmide** quando têm o mesmo vértice e a base do cone está inscrita na base da pirâmide. Na pirâmide regular: o raio do cone é o apótema da base ($r = m$), e a **geratriz do cone é o apótema da pirâmide** ($g_{\text{cone}} = g$), pois o cone toca cada face lateral ao longo do apótema.

Um **cone está circunscrito a uma pirâmide** quando a base da pirâmide está inscrita na base do cone, com o mesmo vértice. Na pirâmide regular: $r = R$ e a **geratriz do cone é a aresta lateral** da pirâmide.

### Prisma e pirâmide; cilindro e cone

Uma **pirâmide inscrita num prisma** tem a mesma base que o prisma e o vértice na outra base; o volume dela é $\frac{1}{3}$ do volume do prisma (notebook 10). O mesmo vale para um **cone inscrito num cilindro**: $\frac{1}{3}$ do volume.

### Cilindro inscrito num cone

O cilindro fica com uma base apoiada na base do cone e a circunferência da outra base encostada na superfície lateral. Na seção meridiana, é um **retângulo dentro de um triângulo**. Se o cone tem raio $R$ e altura $H$, e o cilindro tem raio $r$ e altura $h$, a semelhança entre o triângulo inteiro e o triângulo de cima (acima do cilindro) dá:
$$\frac{r}{R} = \frac{H - h}{H} \qquad\Longrightarrow\qquad r = R\left(1 - \frac{h}{H}\right)$$

Um cilindro baixo e largo ou alto e fino: qual tem o **maior volume**? Você vai descobrir numericamente no Desafio 7.

🕹️ Escolha uma combinação. Nas quatro primeiras, a figura 2D mostra a **base vista de cima**; na última, a seção meridiana. O controle `lados` não vale para a última opção, e `altura do cilindro` vale só para ela.

In [4]:
COMBINACOES = [
    "Cilindro inscrito no prisma", "Cilindro circunscrito ao prisma",
    "Cone inscrito na pirâmide", "Cone circunscrito à pirâmide",
    "Cilindro inscrito no cone",
]


def combinar_solidos(
    combinacao: str, lados: int, lado: float, altura: float, altura_do_cilindro: float
) -> None:
    """Desenha a combinação escolhida em 3D e em 2D e compara os volumes."""
    if combinacao == "Cilindro inscrito no cone":
        cilindro_no_cone(lado, altura, altura_do_cilindro)
        return

    m = lado / (2 * math.tan(math.pi / lados))        # apótema da base
    R_base = lado / (2 * math.sin(math.pi / lados))   # raio da base
    g_piramide = math.sqrt(altura**2 + m**2)          # apótema da pirâmide = geratriz do cone inscrito
    a_piramide = math.sqrt(altura**2 + R_base**2)     # aresta lateral = geratriz do cone circunscrito
    area_da_base = lados * lado * m / 2               # n triângulos de base `lado` e altura m

    inscrito = "inscrito" in combinacao
    eh_cone = combinacao.startswith("Cone")
    raio = m if inscrito else R_base
    if eh_cone:
        volume_poliedro = area_da_base * altura / 3
        volume_redondo = math.pi * raio**2 * altura / 3
    else:
        volume_poliedro = area_da_base * altura
        volume_redondo = math.pi * raio**2 * altura

    # Figura 3D: o poliedro e o sólido redondo, com a base no mesmo plano.
    base = u3d.poligono_regular(lados, lado=lado)
    fig = u3d.criar_figura(titulo=combinacao)
    if eh_cone:
        fig = u3d.desenhar_poliedro(
            fig, *u3d.piramide(base, (0, 0, altura)), cor=COR_FORA, opacidade=0.3
        )
        fig = u3d.desenhar_cone(
            fig, (0, 0, altura), (0, 0, 0), raio, cor=COR_DENTRO, opacidade=0.4
        )
    else:
        fig = u3d.desenhar_poliedro(
            fig, *u3d.prisma(base, (0, 0, altura)), cor=COR_FORA, opacidade=0.3
        )
        fig = u3d.desenhar_cilindro(
            fig, (0, 0, 0), raio, (0, 0, altura), cor=COR_DENTRO, opacidade=0.4
        )
    u3d.mostrar(fig)

    # Figura 2D: a base vista de cima, com a circunferência inscrita ou circunscrita.
    secao = u3d.criar_grafico(titulo="A base vista de cima", proporcao_igual=True)
    secao = u3d.desenhar_poligono_2d(
        secao, [(x, y) for x, y, _ in base], rotulo="base do poliedro", cor=COR_FORA
    )
    secao = u3d.desenhar_circunferencia_2d(
        secao, (0, 0), raio, rotulo="base do sólido redondo", cor=COR_DENTRO,
        preencher=True, opacidade=0.25,
    )
    if inscrito:
        # A circunferência inscrita toca cada lado no ponto médio.
        for k in range(lados):
            x1, y1, _ = base[k]
            x2, y2, _ = base[(k + 1) % lados]
            secao = u3d.marcar_ponto_no_grafico(
                secao, (x1 + x2) / 2, (y1 + y2) / 2, cor=COR_TANGENCIA, tamanho=8
            )
        x1, y1, _ = base[0]
        x2, y2, _ = base[1]
        meio = ((x1 + x2) / 2, (y1 + y2) / 2)
        secao = u3d.desenhar_curva(
            secao, [0, meio[0]], [0, meio[1]], rotulo="apótema m = raio", cor=COR_MEDIDA
        )
    else:
        # A circunferência circunscrita passa pelos vértices.
        for x, y, _ in base:
            secao = u3d.marcar_ponto_no_grafico(secao, x, y, cor=COR_TANGENCIA, tamanho=8)
        secao = u3d.desenhar_curva(
            secao, [0, base[0][0]], [0, base[0][1]], rotulo="raio R da base = raio",
            cor=COR_MEDIDA,
        )
    u3d.mostrar(secao)

    nome_redondo = "cone" if eh_cone else "cilindro"
    nome_poliedro = "pirâmide" if eh_cone else "prisma"
    print(f"Apótema da base: m = {m:.4f};  raio da base: R = {R_base:.4f}")
    print(f"Raio do {nome_redondo}: {raio:.4f} ({'m' if inscrito else 'R'})")
    if eh_cone:
        if inscrito:
            print(f"Geratriz do cone = apótema da pirâmide = √(h² + m²) = {g_piramide:.4f}")
        else:
            print(f"Geratriz do cone = aresta lateral da pirâmide = √(h² + R²) = {a_piramide:.4f}")
    print(f"Volume da {nome_poliedro}: {volume_poliedro:.4f}" if eh_cone
          else f"Volume do {nome_poliedro}: {volume_poliedro:.4f}")
    print(f"Volume do {nome_redondo}: {volume_redondo:.4f}")
    print(f"Razão volume do {nome_redondo} / volume do poliedro: "
          f"{volume_redondo / volume_poliedro:.4f}")


def cilindro_no_cone(raio_do_cone: float, altura_do_cone: float, h: float) -> None:
    """Desenha o cilindro inscrito no cone, em 3D e na seção meridiana."""
    R, H = raio_do_cone, altura_do_cone
    r_cilindro_no_cone = R * (1 - h / H)       # semelhança: r / R = (H − h) / H
    volume_cone = math.pi * R**2 * H / 3
    volume_cilindro = math.pi * r_cilindro_no_cone**2 * h

    fig = u3d.criar_figura(titulo="Cilindro inscrito no cone")
    fig = u3d.desenhar_cone(fig, (0, 0, H), (0, 0, 0), R, cor=COR_FORA, opacidade=0.25)
    fig = u3d.desenhar_cilindro(
        fig, (0, 0, 0), r_cilindro_no_cone, (0, 0, h), cor=COR_DENTRO, opacidade=0.5
    )
    u3d.mostrar(fig)

    secao = u3d.criar_grafico(titulo="Seção meridiana: retângulo no triângulo", proporcao_igual=True)
    secao = u3d.desenhar_poligono_2d(
        secao, [(-R, 0), (R, 0), (0, H)], rotulo="cone", cor=COR_FORA, opacidade=0.15
    )
    secao = u3d.desenhar_poligono_2d(
        secao,
        [(-r_cilindro_no_cone, 0), (r_cilindro_no_cone, 0),
         (r_cilindro_no_cone, h), (-r_cilindro_no_cone, h)],
        rotulo="cilindro", cor=COR_DENTRO,
    )
    # Os dois triângulos retângulos semelhantes: o "grande" (metade do cone) e o
    # "de cima" (acima do cilindro). Eles têm o ângulo do vértice em comum.
    secao = u3d.desenhar_poligono_2d(
        secao, [(0, 0), (R, 0), (0, H)], rotulo="triângulo grande (R, H)",
        cor=u3d.CORES["plano_terciario"], preencher=False, espessura=3,
    )
    secao = u3d.desenhar_poligono_2d(
        secao, [(0, h), (r_cilindro_no_cone, h), (0, H)],
        rotulo="triângulo de cima (r, H − h)", cor=COR_MEDIDA, opacidade=0.5,
    )
    secao = u3d.marcar_ponto_no_grafico(
        secao, r_cilindro_no_cone, h, rotulo="P", cor=COR_TANGENCIA
    )
    u3d.mostrar(secao)

    print(f"Raio do cilindro: r = R · (1 − h/H) = {R:g} · (1 − {h:g}/{H:g}) "
          f"= {r_cilindro_no_cone:.4f}")
    print(f"Volume do cone: {volume_cone:.4f}")
    print(f"Volume do cilindro: {volume_cilindro:.4f}")
    print(f"Razão volume do cilindro / volume do cone: {volume_cilindro / volume_cone:.4f}")


controle_combinacao = widgets.Dropdown(
    options=COMBINACOES, value=COMBINACOES[0], description="combinação",
    layout=widgets.Layout(width="max-content"),
)
controle_lados = widgets.Dropdown(options=[3, 4, 6], value=6, description="lados")
controle_lado = widgets.FloatSlider(
    min=1, max=4, step=0.5, value=2, description="lado (ou R)", continuous_update=False,
)
controle_altura = widgets.FloatSlider(
    min=2, max=8, step=0.5, value=5, description="altura", continuous_update=False,
)
controle_altura_do_cilindro = widgets.FloatSlider(
    min=0.5, max=4.5, step=0.5, value=2, description="altura do cilindro",
    style={"description_width": "initial"}, continuous_update=False,
)


def ajustar_controles(_=None) -> None:
    """A altura do cilindro fica sempre menor que a do cone; cada controle só vale
    para as combinações em que faz sentido."""
    controle_altura_do_cilindro.max = controle_altura.value - 0.5
    no_cone = controle_combinacao.value == "Cilindro inscrito no cone"
    controle_altura_do_cilindro.disabled = not no_cone
    controle_lados.disabled = no_cone


controle_altura.observe(ajustar_controles, names="value")
controle_combinacao.observe(ajustar_controles, names="value")
ajustar_controles()

interact(
    combinar_solidos,
    combinacao=controle_combinacao,
    lados=controle_lados,
    lado=controle_lado,
    altura=controle_altura,
    altura_do_cilindro=controle_altura_do_cilindro,
);

interactive(children=(Dropdown(description='combinação', layout=Layout(width='max-content'), options=('Cilindr…

**Pare e pense**

1. Um prisma hexagonal regular tem aresta da base 2 e altura 5. Qual é o volume do cilindro inscrito? E do circunscrito?
2. Num cone de raio 3 e altura 6, um cilindro inscrito tem altura 2. Qual é o raio do cilindro?

<details>
<summary>Ver respostas</summary>

1. Inscrito: $r = m = \sqrt{3}$, volume $= \pi \cdot 3 \cdot 5 = 15\pi \approx 47{,}1$. Circunscrito: $r = 2$, volume $= 20\pi \approx 62{,}8$.
2. $r = 3\left(1 - \frac{2}{6}\right) = 2$.
</details>

## 3. Cilindro e esfera

### Cilindro inscrito numa esfera

As duas circunferências das bases do cilindro ficam sobre a superfície da esfera. Na seção meridiana, é um **retângulo inscrito num círculo**: a diagonal do retângulo é um diâmetro. Para um cilindro de raio $r$ e altura $h$, numa esfera de raio $R$:
$$(2r)^2 + h^2 = (2R)^2 \qquad\Longleftrightarrow\qquad r^2 + \left(\frac{h}{2}\right)^2 = R^2$$

### Esfera inscrita num cilindro

Para a esfera tocar as duas bases **e** a superfície lateral, a seção meridiana precisa ter um círculo tangente aos quatro lados do retângulo: o retângulo tem de ser um **quadrado**. Então:

> Só o **cilindro equilátero** ($h = 2r$) admite esfera inscrita, e o raio da esfera é igual ao raio do cilindro.

É o par de Arquimedes do notebook 13: a esfera tem $\frac{2}{3}$ do volume e da área total desse cilindro.

Inverta a pergunta: dentre todos os cilindros inscritos numa esfera, qual tem o maior volume? 

🕹️ Explore no widget, com o gráfico do volume em função da altura.

In [5]:
def cilindro_na_esfera(raio_da_esfera: float, altura_do_cilindro: float) -> None:
    """Desenha o cilindro inscrito na esfera e o gráfico do volume em função da altura."""
    R, h = raio_da_esfera, altura_do_cilindro
    r = math.sqrt(raio_da_esfera**2 - (altura_do_cilindro / 2)**2)
    volume_cilindro = math.pi * r**2 * altura_do_cilindro
    volume_esfera = 4 / 3 * math.pi * raio_da_esfera**3

    # Figura 3D: a esfera com centro na origem e o cilindro centrado nela.
    fig = u3d.criar_figura(titulo="Cilindro inscrito na esfera")
    fig = u3d.desenhar_esfera(fig, (0, 0, 0), R, cor=COR_FORA, opacidade=0.15)
    fig = u3d.desenhar_cilindro(fig, (0, 0, -h / 2), r, (0, 0, h), cor=COR_DENTRO, opacidade=0.5)
    u3d.mostrar(fig)

    # Seção meridiana: o retângulo 2r × h inscrito no círculo máximo.
    secao = u3d.criar_grafico(titulo="Seção meridiana: retângulo no círculo", proporcao_igual=True)
    secao = u3d.desenhar_circunferencia_2d(
        secao, (0, 0), R, rotulo="esfera", cor=COR_FORA, preencher=True, opacidade=0.1
    )
    secao = u3d.desenhar_poligono_2d(
        secao, [(-r, -h / 2), (r, -h / 2), (r, h / 2), (-r, h / 2)],
        rotulo="cilindro", cor=COR_DENTRO,
    )
    secao = u3d.desenhar_curva(
        secao, [-r, r], [-h / 2, h / 2], rotulo="diagonal = diâmetro 2R", cor=COR_MEDIDA
    )
    secao = u3d.marcar_ponto_no_grafico(secao, 0, 0, rotulo="O", cor=u3d.CORES["apoio"])
    for x, y in [(-r, -h / 2), (r, -h / 2), (r, h / 2), (-r, h / 2)]:
        secao = u3d.marcar_ponto_no_grafico(secao, x, y, cor=COR_TANGENCIA, tamanho=8)
    u3d.mostrar(secao)

    # Gráfico extra: o volume do cilindro para cada altura possível, de 0 a 2R.
    alturas = np.linspace(0, 2 * R, 201)
    volumes = []
    for altura in alturas:
        raio = math.sqrt(max(R**2 - (altura / 2)**2, 0))   # max(..., 0) evita −0,000…
        volumes.append(math.pi * raio**2 * altura)
    grafico = u3d.criar_grafico(
        titulo="Volume do cilindro inscrito × altura", rotulo_x="altura h",
        rotulo_y="volume do cilindro",
    )
    grafico = u3d.desenhar_curva(grafico, alturas, volumes, cor=COR_DENTRO)
    grafico = u3d.marcar_ponto_no_grafico(
        grafico, h, volume_cilindro, rotulo="cilindro atual", cor=COR_TANGENCIA
    )
    u3d.mostrar(grafico)

    print(f"Raio do cilindro: r = √(R² − (h/2)²) = {r:.4f}")
    print(f"Volume do cilindro: {volume_cilindro:.4f}")
    print(f"Volume da esfera: {volume_esfera:.4f}")
    print(f"O cilindro ocupa {volume_cilindro / volume_esfera * 100:.1f}% da esfera.")


controle_raio_da_esfera = widgets.FloatSlider(
    min=1, max=3, step=0.5, value=2, description="raio da esfera",
    style={"description_width": "initial"}, continuous_update=False,
)
controle_altura_na_esfera = widgets.FloatSlider(
    min=0.2, max=3.8, step=0.2, value=2, description="altura do cilindro",
    style={"description_width": "initial"}, continuous_update=False,
)


def limitar_altura_na_esfera(_=None) -> None:
    """A altura do cilindro fica sempre menor que o diâmetro da esfera."""
    controle_altura_na_esfera.max = round(2 * controle_raio_da_esfera.value - 0.2, 1)


controle_raio_da_esfera.observe(limitar_altura_na_esfera, names="value")
limitar_altura_na_esfera()

interact(
    cilindro_na_esfera,
    raio_da_esfera=controle_raio_da_esfera,
    altura_do_cilindro=controle_altura_na_esfera,
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='raio da esfera', max=3.0, m…

**Pare e pense**

1. Um cilindro de altura 6 está inscrito numa esfera de raio 5. Qual é o raio do cilindro e o seu volume?
2. Pelo gráfico, o cilindro de maior volume é o equilátero? (Compare a altura do máximo com $2r$.)

<details>
<summary>Ver respostas</summary>

1. $r = \sqrt{25 - 9} = 4$; volume $= \pi \cdot 16 \cdot 6 = 96\pi \approx 301{,}6$.
2. Não. O máximo acontece com $h = \frac{2R}{\sqrt{3}} \approx 1{,}15R$, e nesse ponto $h = r\sqrt{2}$, menor que $2r$. O cilindro equilátero é o que "abraça" a esfera por fora, não o maior que cabe dentro dela.
</details>

## 4. Esfera e cone reto

Considere um cone reto de raio $R$, altura $H$ e geratriz $g$. A seção meridiana é um triângulo isósceles de base $2R$, altura $H$ e lados $g$.

### Esfera inscrita no cone

A esfera toca a base e a superfície lateral. Na seção meridiana, é a **circunferência inscrita** no triângulo. Usando $r = \frac{\text{área}}{\text{semiperímetro}}$:
$$\text{área} = \frac{2R \cdot H}{2} = RH \qquad \text{semiperímetro} = \frac{2R + 2g}{2} = R + g$$
$$\boxed{r = \frac{R\,H}{R + g}}$$

### Esfera circunscrita ao cone

A esfera passa pelo vértice e pela circunferência da base. Na seção meridiana, é a **circunferência circunscrita** ao triângulo. O centro fica no eixo do cone, a uma distância $R_e$ do vértice; portanto, a uma distância $|H - R_e|$ do centro da base. Pelo triângulo retângulo formado pelo centro da esfera, o centro da base e um ponto da borda da base:
$$(H - R_e)^2 + R^2 = R_e^2 \qquad\Longrightarrow\qquad \boxed{R_e = \frac{H^2 + R^2}{2H} = \frac{g^2}{2H}}$$

Se o cone for "achatado" ($H < R$), o centro da esfera circunscrita fica **abaixo** da base do cone.

🕹️ Escolha a esfera e mexa nas medidas do cone.

In [6]:
def esfera_e_cone(esfera: str, raio_do_cone: float, altura_do_cone: float) -> None:
    """Desenha o cone com a esfera inscrita ou circunscrita, em 3D e em 2D."""
    R, H = raio_do_cone, altura_do_cone
    g = math.sqrt(altura_do_cone**2 + raio_do_cone**2)
    r_inscrita = raio_do_cone * altura_do_cone / (raio_do_cone + g)
    R_circunscrita = g**2 / (2 * altura_do_cone)
    centro_da_circunscrita = altura_do_cone - R_circunscrita   # altura do centro, medida a partir da base

    inscrita = esfera == "Inscrita"
    if inscrita:
        centro, raio = r_inscrita, r_inscrita
    else:
        centro, raio = centro_da_circunscrita, R_circunscrita

    fig = u3d.criar_figura(titulo=f"Esfera {esfera.lower()} no cone")
    fig = u3d.desenhar_cone(
        fig, (0, 0, H), (0, 0, 0), R, cor=COR_FORA, opacidade=0.25 if inscrita else 0.5
    )
    fig = u3d.desenhar_esfera(
        fig, (0, 0, centro), raio, cor=COR_DENTRO, opacidade=0.5 if inscrita else 0.15
    )
    u3d.mostrar(fig)

    vertices = [(-R, 0), (R, 0), (0, H)]
    secao = u3d.criar_grafico(
        titulo=f"Seção meridiana: circunferência {esfera.lower()} ao triângulo",
        proporcao_igual=True,
    )
    secao = u3d.desenhar_poligono_2d(secao, vertices, rotulo="cone", cor=COR_FORA)
    secao = u3d.desenhar_circunferencia_2d(
        secao, (0, centro), raio, rotulo="esfera", cor=COR_DENTRO, preencher=inscrita
    )
    secao = u3d.marcar_ponto_no_grafico(secao, 0, centro, rotulo="O", cor=u3d.CORES["apoio"])
    if inscrita:
        # Pontos de tangência: o centro da base e, em cada lado, o ponto a uma
        # distância R do canto da base (dois segmentos tangentes traçados de um mesmo
        # ponto têm o mesmo comprimento, módulo 04).
        T_direita = (R - R * R / g, R * H / g)
        tangencias = [(0, 0), T_direita, (-T_direita[0], T_direita[1])]
        for x, y in tangencias:
            secao = u3d.marcar_ponto_no_grafico(secao, x, y, cor=COR_TANGENCIA, tamanho=8)
        secao = u3d.desenhar_curva(
            secao, [0, T_direita[0]], [centro, T_direita[1]], rotulo="raio r", cor=COR_MEDIDA
        )
    else:
        for x, y in vertices:
            secao = u3d.marcar_ponto_no_grafico(secao, x, y, cor=COR_TANGENCIA, tamanho=8)
        # O triângulo retângulo da conta: centro da esfera, centro da base, borda da base.
        secao = u3d.desenhar_poligono_2d(
            secao, [(0, centro), (0, 0), (R, 0)], rotulo="(H − Rₑ)² + R² = Rₑ²",
            cor=COR_MEDIDA, opacidade=0.4,
        )
    u3d.mostrar(secao)

    print(f"Geratriz: g = √(H² + R²) = {g:.4f}")
    print(f"Esfera inscrita: r = RH / (R + g) = {r_inscrita:.4f}")
    print(f"Esfera circunscrita: Rₑ = g² / (2H) = {R_circunscrita:.4f}")
    print(f"Altura do centro da circunscrita, medida a partir da base: H − Rₑ = "
          f"{centro_da_circunscrita:.4f}")
    if altura_do_cone < raio_do_cone:
        print("Como H < R (cone achatado), o centro da esfera circunscrita está ABAIXO da base.")

    # Conferência com o utils_3d, aplicada ao triângulo da seção meridiana.
    centro_i, raio_i = u3d.medir_circunferencia_inscrita(*vertices)
    centro_c, raio_c = u3d.medir_circunferencia_circunscrita(*vertices)
    print(f"Conferência com o utils_3d: r = {raio_i:.4f} (centro a {centro_i[1]:.4f} da base), "
          f"Rₑ = {raio_c:.4f} (centro a {centro_c[1]:.4f})")


interact(
    esfera_e_cone,
    esfera=widgets.RadioButtons(options=["Inscrita", "Circunscrita"], value="Inscrita"),
    raio_do_cone=widgets.FloatSlider(
        min=1, max=4, step=0.5, value=3, description="raio do cone",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    altura_do_cone=widgets.FloatSlider(
        min=1, max=8, step=0.5, value=4, description="altura do cone",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(RadioButtons(description='esfera', options=('Inscrita', 'Circunscrita'), value='Inscrita…

**Pare e pense**

1. Para o cone de raio 3 e altura 4, calcule os raios das esferas inscrita e circunscrita.
2. Para o mesmo cone, qual fração do volume do cone a esfera inscrita ocupa?

<details>
<summary>Ver respostas</summary>

1. $g = 5$; $r = \frac{12}{8} = 1{,}5$; $R_e = \frac{25}{8} = 3{,}125$.
2. Volume da esfera: $\frac{4}{3}\pi \cdot 1{,}5^3 = 4{,}5\pi$; volume do cone: $12\pi$. Fração: $\frac{4{,}5}{12} = 0{,}375$.
</details>

## 5. Esfera, cilindro equilátero e cone equilátero

Considere uma esfera de raio $R$ e dois sólidos **circunscritos** a ela:
- o **cilindro equilátero** circunscrito: raio $R$ e altura $2R$;
- o **cone equilátero** circunscrito: a seção meridiana é um triângulo equilátero com a circunferência de raio $R$ inscrita. No triângulo equilátero, o raio inscrito é $\frac{1}{3}$ da altura; então o cone tem altura $3R$ e raio da base $R\sqrt{3}$ (geratriz $2R\sqrt{3}$).

Agora compare:

| Sólido | Volume | Área total |
|---|---|---|
| esfera | $\frac{4}{3}\pi R^3$ | $4\pi R^2$ |
| cilindro equilátero | $2\pi R^3$ | $6\pi R^2$ |
| cone equilátero | $3\pi R^3$ | $9\pi R^2$ |

Multiplicando por 3 os volumes, ou simplesmente olhando as áreas:
$$\boxed{\text{esfera} : \text{cilindro} : \text{cone} = 4 : 6 : 9}$$

**A mesma proporção vale para volumes e para áreas!** E repare: $6^2 = 4 \cdot 9$. O cilindro é a **média geométrica** entre a esfera e o cone.

> 📎 **Conexão.** A relação entre a esfera e o cilindro (4 : 6, ou seja, 2 : 3) é a descoberta de Arquimedes do notebook 13. O cone equilátero completa o trio.

🕹️ Veja os três juntos.

In [7]:
def trio(raio_da_esfera: float, cilindro: bool, cone: bool) -> None:
    """Desenha a esfera com o cilindro e o cone equiláteros circunscritos e compara."""
    R = raio_da_esfera
    volume_esfera = 4 / 3 * math.pi * R**3
    volume_cilindro = math.pi * R**2 * (2 * R)
    raio_do_cone = R * math.sqrt(3)
    altura_do_cone = 3 * R
    volume_cone = math.pi * raio_do_cone**2 * altura_do_cone / 3
    geratriz_do_cone = 2 * raio_do_cone
    area_esfera = 4 * math.pi * R**2
    area_cilindro = 2 * math.pi * R * (2 * R) + 2 * math.pi * R**2
    area_cone = math.pi * raio_do_cone * geratriz_do_cone + math.pi * raio_do_cone**2

    # Os três sólidos ficam apoiados no mesmo plano (a base em z = 0).
    cor_cone = u3d.CORES["plano_terciario"]
    fig = u3d.criar_figura(titulo="Esfera, cilindro equilátero e cone equilátero")
    fig = u3d.desenhar_esfera(fig, (0, 0, R), R, cor=COR_DENTRO, opacidade=0.9)
    if cilindro:
        fig = u3d.desenhar_cilindro(fig, (0, 0, 0), R, (0, 0, 2 * R), cor=COR_FORA, opacidade=0.2)
    if cone:
        fig = u3d.desenhar_cone(
            fig, (0, 0, altura_do_cone), (0, 0, 0), raio_do_cone, cor=cor_cone, opacidade=0.15
        )
    u3d.mostrar(fig)

    secao = u3d.criar_grafico(
        titulo="Seção meridiana: círculo no quadrado e no triângulo equilátero",
        proporcao_igual=True,
    )
    if cone:
        secao = u3d.desenhar_poligono_2d(
            secao, [(-raio_do_cone, 0), (raio_do_cone, 0), (0, altura_do_cone)],
            rotulo="cone equilátero", cor=cor_cone, opacidade=0.15,
        )
    if cilindro:
        secao = u3d.desenhar_poligono_2d(
            secao, [(-R, 0), (R, 0), (R, 2 * R), (-R, 2 * R)],
            rotulo="cilindro equilátero", cor=COR_FORA, opacidade=0.2,
        )
    secao = u3d.desenhar_circunferencia_2d(
        secao, (0, R), R, rotulo="esfera", cor=COR_DENTRO, preencher=True, opacidade=0.5
    )
    u3d.mostrar(secao)

    print(f"{'sólido':<22}{'volume':>10}{'área total':>12}{'volume × 4/esfera':>20}"
          f"{'área × 4/esfera':>18}")
    for nome, volume, area in [
        ("esfera", volume_esfera, area_esfera),
        ("cilindro equilátero", volume_cilindro, area_cilindro),
        ("cone equilátero", volume_cone, area_cone),
    ]:
        print(f"{nome:<22}{volume:>10.4f}{area:>12.4f}{volume / volume_esfera * 4:>20.4f}"
              f"{area / area_esfera * 4:>18.4f}")


interact(
    trio,
    raio_da_esfera=widgets.FloatSlider(
        min=0.5, max=2, step=0.25, value=1, description="raio da esfera",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    cilindro=widgets.Checkbox(value=True, description="Cilindro"),
    cone=widgets.Checkbox(value=True, description="Cone"),
);

interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='raio da esfera', max=2.0, m…

**Pare e pense**

1. Para uma esfera de raio 2, calcule os volumes do cilindro equilátero e do cone equilátero circunscritos.
2. Confirme que as razões dão 4 : 6 : 9.

<details>
<summary>Ver respostas</summary>

1. Esfera: $\frac{32\pi}{3}$. Cilindro: $16\pi$. Cone: raio $2\sqrt{3}$ e altura 6, volume $\frac{1}{3}\pi \cdot 12 \cdot 6 = 24\pi$.
2. $\frac{32}{3} : 16 : 24 = 32 : 48 : 72 = 4 : 6 : 9$.
</details>

## 6. Esfera e tronco de cone

Uma esfera de raio $\rho$ está **inscrita num tronco de cone** reto (raios $R$ e $r$, altura $h$, geratriz $g$) quando toca as duas bases e a superfície lateral. Na seção meridiana, é uma **circunferência inscrita num trapézio isósceles**.

**Duas consequências:**
1. A altura do tronco é o diâmetro da esfera: $h = 2\rho$.
2. Num quadrilátero com circunferência inscrita, as somas dos lados opostos são iguais (módulo 04). No trapézio: $2R + 2r = 2g$, ou seja,
$$\boxed{g = R + r}$$

Juntando com o Pitágoras do tronco ($g^2 = h^2 + (R - r)^2$, notebook 14):
$$h^2 = (R + r)^2 - (R - r)^2 = 4Rr \qquad\Longrightarrow\qquad h = 2\sqrt{Rr} \qquad\Longrightarrow\qquad \boxed{\rho = \sqrt{R\,r}}$$

O raio da esfera é a **média geométrica** dos raios das bases!

> ⚠️ **Erro comum.** Achar que todo tronco de cone tem esfera inscrita. Só os que satisfazem $g = R + r$ (equivalentemente, $h = 2\sqrt{Rr}$).

🕹️ Escolha os raios do tronco: a altura é ajustada para que a esfera caiba.

In [8]:
def esfera_no_tronco(raio_maior: float, raio_menor: float) -> None:
    """Desenha o tronco de cone com a esfera inscrita, em 3D e na seção meridiana."""
    rho = math.sqrt(raio_maior * raio_menor)
    altura = 2 * rho
    geratriz = raio_maior + raio_menor
    conferencia = math.sqrt(altura**2 + (raio_maior - raio_menor)**2)   # deve dar igual à geratriz
    volume_tronco = math.pi * altura / 3 * (raio_maior**2 + raio_maior * raio_menor + raio_menor**2)
    volume_esfera = 4 / 3 * math.pi * rho**3

    fig = u3d.criar_figura(titulo="Esfera inscrita no tronco de cone")
    fig = u3d.desenhar_tronco_de_cone(
        fig, (0, 0, 0), raio_maior, raio_menor, altura, cor=COR_FORA, opacidade=0.25
    )
    fig = u3d.desenhar_esfera(fig, (0, 0, rho), rho, cor=COR_DENTRO, opacidade=0.5)
    u3d.mostrar(fig)

    R, r = raio_maior, raio_menor
    secao = u3d.criar_grafico(
        titulo="Seção meridiana: circunferência no trapézio", proporcao_igual=True
    )
    secao = u3d.desenhar_poligono_2d(
        secao, [(-R, 0), (R, 0), (r, altura), (-r, altura)], rotulo="tronco", cor=COR_FORA
    )
    secao = u3d.desenhar_circunferencia_2d(
        secao, (0, rho), rho, rotulo="esfera", cor=COR_DENTRO, preencher=True
    )
    # Pontos de tangência: o meio de cada base e, em cada lado, o ponto a uma distância
    # R do canto de baixo (os segmentos tangentes traçados desse canto medem R).
    T_direita = (R + R * (r - R) / geratriz, R * altura / geratriz)
    tangencias = [(0, 0), (0, altura), T_direita, (-T_direita[0], T_direita[1])]
    for x, y in tangencias:
        secao = u3d.marcar_ponto_no_grafico(secao, x, y, cor=COR_TANGENCIA, tamanho=8)
    secao = u3d.desenhar_curva(
        secao, [0, 0], [0, altura], rotulo="altura h = 2ρ", cor=COR_MEDIDA, tracejado=True
    )
    u3d.mostrar(secao)

    print(f"Raio da esfera: ρ = √(Rr) = {rho:.4f}")
    print(f"Altura do tronco: h = 2ρ = {altura:.4f}")
    print(f"Geratriz: g = R + r = {geratriz:.4f};  conferência √(h² + (R − r)²) = {conferencia:.4f}")
    print(f"Volume do tronco: {volume_tronco:.4f}")
    print(f"Volume da esfera: {volume_esfera:.4f}")
    print(f"A esfera ocupa {volume_esfera / volume_tronco * 100:.1f}% do tronco.")


controle_raio_maior = widgets.FloatSlider(
    min=2, max=9, step=0.5, value=9, description="raio maior R",
    style={"description_width": "initial"}, continuous_update=False,
)
controle_raio_menor = widgets.FloatSlider(
    min=0.5, max=8.5, step=0.5, value=4, description="raio menor r",
    style={"description_width": "initial"}, continuous_update=False,
)


def limitar_raio_menor(_=None) -> None:
    """O raio menor fica sempre menor que o raio maior."""
    controle_raio_menor.max = controle_raio_maior.value - 0.5


controle_raio_maior.observe(limitar_raio_menor, names="value")
limitar_raio_menor()

interact(esfera_no_tronco, raio_maior=controle_raio_maior, raio_menor=controle_raio_menor);

interactive(children=(FloatSlider(value=9.0, continuous_update=False, description='raio maior R', max=9.0, min…

**Pare e pense**

1. Um tronco de cone circunscrito a uma esfera tem raios das bases 9 e 4. Quais são o raio da esfera, a altura e a geratriz do tronco?
2. O que acontece com a fórmula $\rho = \sqrt{Rr}$ quando $R = r$? Que sólido é esse?

<details>
<summary>Ver respostas</summary>

1. $\rho = \sqrt{36} = 6$; $h = 12$; $g = 13$.
2. $\rho = R$: o "tronco" é um cilindro com $h = 2R$, o cilindro equilátero da seção 3.
</details>

## Resumo

**Técnica:** corte pelo eixo (seção meridiana). Cilindro → retângulo; cone → triângulo isósceles; tronco de cone → trapézio isósceles; esfera → círculo máximo.

**Prisma regular e cilindro:** inscrito $r = m$ (apótema da base); circunscrito $r = R$ (raio da base).

**Pirâmide regular e cone:** inscrito $r = m$ e geratriz = apótema da pirâmide; circunscrito $r = R$ e geratriz = aresta lateral.

**Cilindro inscrito no cone** ($R$, $H$): $r = R\left(1 - \frac{h}{H}\right)$.

**Cilindro inscrito na esfera:** $r^2 + \left(\frac{h}{2}\right)^2 = R^2$. **Esfera inscrita no cilindro:** só no equilátero, com raio igual ao do cilindro.

**Esfera e cone** ($R$, $H$, $g$): inscrita $r = \dfrac{RH}{R + g}$; circunscrita $R_e = \dfrac{g^2}{2H}$.

**Esfera, cilindro equilátero e cone equilátero circunscritos:** volumes e áreas na razão $4 : 6 : 9$.

**Esfera inscrita no tronco de cone:** $g = R + r$, $h = 2\rho$, $\rho = \sqrt{Rr}$.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Desenhe a seção meridiana antes de calcular! Use `math.pi` e `math.sqrt`.

O gabarito comentado está em [`0616b_inscricao_e_circunscricao_corpos_redondos_gabarito.ipynb`](0616b_inscricao_e_circunscricao_corpos_redondos_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) O cilindro inscrito num prisma quadrangular regular tem raio igual à metade do
#     lado da base.
# (b) Só o cilindro equilátero admite esfera inscrita.
# (c) No cone inscrito numa pirâmide regular, a geratriz do cone é igual ao apótema
#     da pirâmide.
# (d) Num tronco de cone com esfera inscrita, a geratriz é a diferença entre os raios
#     das bases.
# (e) Esfera, cilindro equilátero e cone equilátero circunscritos a ela têm volumes
#     na razão 4 : 6 : 9.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": True, "c": True, "d": False, "e": True}
dicas_ex1 = {
    "a": "o raio do cilindro inscrito é o apótema da base, que no quadrado é metade do lado (seção 2 - Prismas, pirâmides, cilindros e cones)",
    "b": "a seção meridiana precisa ser um quadrado (seção 3 - Cilindro e esfera)",
    "c": "o cone toca cada face lateral ao longo do apótema (seção 2 - Prismas, pirâmides, cilindros e cones)",
    "d": "no trapézio com circunferência inscrita, as somas dos lados opostos são iguais (seção 6 - Esfera e tronco de cone)",
    "e": "compare as três linhas da tabela de volumes (seção 5 - Esfera, cilindro equilátero e cone equilátero)",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex1[item]}."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: esferas e cone. Para o cone de raio 3 e altura 4, calcule:
# (a) o raio da esfera inscrita;
# (b) o raio da esfera circunscrita.
r_ex2 = ...
R_ex2 = ...

print(r_ex2, R_ex2)

In [ ]:
# Verificação
respostas_ex2 = [("a", r_ex2, 1.5), ("b", R_ex2, 3.125)]
for item, resposta, certa in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), r = RH / (R + g) e R_e = g² / (2H), "
        "com g = 5 (seção 4 - Esfera e cone reto)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: o trio. Para uma esfera de raio 2, calcule:
# (a) o volume da esfera;
# (b) o volume do cilindro equilátero circunscrito a ela;
# (c) o volume do cone equilátero circunscrito a ela.
volume_esfera_ex3 = ...
volume_cilindro_ex3 = ...
volume_cone_ex3 = ...

print(volume_esfera_ex3, volume_cilindro_ex3, volume_cone_ex3)

In [ ]:
# Verificação
respostas_ex3 = [
    ("a", volume_esfera_ex3, 32 * math.pi / 3),
    ("b", volume_cilindro_ex3, 16 * math.pi),
    ("c", volume_cone_ex3, 24 * math.pi),
]
for item, resposta, certa in respostas_ex3:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), o cilindro tem raio R e altura 2R; "
        "o cone equilátero circunscrito tem altura 3R e raio R√3 "
        "(seção 5 - Esfera, cilindro equilátero e cone equilátero)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: prisma hexagonal e cilindros. Um prisma hexagonal regular
# tem aresta da base 2 e altura 5. Calcule os volumes dos cilindros:
# (a) inscrito no prisma;
# (b) circunscrito ao prisma.
volume_inscrito_ex4 = ...
volume_circunscrito_ex4 = ...

print(volume_inscrito_ex4, volume_circunscrito_ex4)

In [ ]:
# Verificação
respostas_ex4 = [
    ("a", volume_inscrito_ex4, 15 * math.pi),
    ("b", volume_circunscrito_ex4, 20 * math.pi),
]
for item, resposta, certa in respostas_ex4:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), inscrito: raio = apótema do hexágono "
        "(√3); circunscrito: raio = lado (2) (seção 2 - Prismas, pirâmides, cilindros e cones)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 5: cilindro na esfera. Um cilindro de altura 6 está inscrito
# numa esfera de raio 5. Calcule:
# (a) o raio do cilindro;
# (b) o volume do cilindro.
r_ex5 = ...
volume_ex5 = ...

print(r_ex5, volume_ex5)

In [ ]:
# Verificação
respostas_ex5 = [("a", r_ex5, 4), ("b", volume_ex5, 96 * math.pi)]
for item, resposta, certa in respostas_ex5:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), na seção meridiana, a diagonal do "
        "retângulo é o diâmetro: r² + 3² = 5² (seção 3 - Cilindro e esfera)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 6: o tronco circunscrito. Um tronco de cone circunscrito a
# uma esfera tem raios das bases 9 e 4. Calcule:
# (a) o raio da esfera;
# (b) a altura do tronco;
# (c) o volume do tronco.
rho_ex6 = ...
altura_ex6 = ...
volume_ex6 = ...

print(rho_ex6, altura_ex6, volume_ex6)

In [ ]:
# Verificação
respostas_ex6 = [
    ("a", rho_ex6, 6),
    ("b", altura_ex6, 12),
    ("c", volume_ex6, 532 * math.pi),
]
for item, resposta, certa in respostas_ex6:
    assert resposta is not ... and abs(resposta - certa) <= 0.1, (
        f"Tente de novo, revise: no item ({item}), ρ = √(Rr), h = 2ρ; "
        "volume = πh/3 · (R² + Rr + r²) (seção 6 - Esfera e tronco de cone)."
    )
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: o maior cilindro dentro do cone. Num cone de raio 3 e altura 6,
# teste as alturas de cilindro inscrito de 0,01 a 5,99 (de 0,01 em 0,01) e encontre a
# altura que dá o MAIOR volume. Guarde:
# (a) essa altura em melhor_altura_ex7;
# (b) a razão entre o volume desse cilindro e o volume do cone em razao_ex7.
R_cone, H_cone = 3, 6
alturas = [k / 100 for k in range(1, 600)]   # 0,01; 0,02; ...; 5,99
volumes = []
for h in alturas:
    ...  # complete aqui: calcule o raio r do cilindro de altura h (semelhança)
    ...  # complete aqui: calcule o volume do cilindro e acrescente-o à lista `volumes`

melhor_altura_ex7 = ...  # dica: o maior volume é max(volumes), na posição volumes.index(...)
razao_ex7 = ...

print(melhor_altura_ex7, razao_ex7)

In [ ]:
# Verificação
assert melhor_altura_ex7 is not ... and abs(melhor_altura_ex7 - 2) <= 0.02, (
    "Tente de novo, revise: para cada h, r = 3 · (1 − h/6) e volume = π r² h "
    "(seção 2 - Prismas, pirâmides, cilindros e cones)."
)
assert razao_ex7 is not ... and abs(razao_ex7 - 4 / 9) <= 0.01, (
    "Tente de novo, revise: divida o maior volume pelo volume do cone, π · 3² · 6 / 3 "
    "(seção 2 - Prismas, pirâmides, cilindros e cones)."
)
print("O resultado vale para qualquer cone: o maior cilindro inscrito tem um terço da")
print("altura do cone e ocupa 4/9 do volume. Provar isso em geral é um problema clássico")
print("de Cálculo (módulo 11).")
print("Certo!")

In [ ]:
# Exercício Desafio 8: o maior cilindro dentro da esfera. Numa esfera de raio 1,
# encontre numericamente a altura do cilindro inscrito de maior volume, testando as
# alturas de 0,001 a 1,999 (de 0,001 em 0,001).
alturas = [k / 1000 for k in range(1, 2000)]   # 0,001; 0,002; ...; 1,999
volumes = []
for h in alturas:
    ...  # complete aqui: calcule r² pela relação da seção meridiana
    ...  # complete aqui: calcule o volume do cilindro e acrescente-o à lista `volumes`

melhor_altura_ex8 = ...

print(melhor_altura_ex8)

In [ ]:
# Verificação
assert melhor_altura_ex8 is not ... and abs(melhor_altura_ex8 - 2 / math.sqrt(3)) <= 0.01, (
    "Tente de novo, revise: para cada h, r² = 1 − (h/2)² e volume = π r² h "
    "(seção 3 - Cilindro e esfera)."
)
print("Certo!")

## Conexões

**Isso depende de:**
- [`0610a_piramide.ipynb`](0610a_piramide.ipynb) (apótema da base, apótema da pirâmide e aresta lateral; a pirâmide inscrita no prisma)
- [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb), [`0612a_cone.ipynb`](0612a_cone.ipynb) e [`0613a_esfera.ipynb`](0613a_esfera.ipynb) (seções meridianas, cilindro e cone equiláteros, os resultados de Arquimedes)
- [`0614a_solidos_semelhantes_e_troncos.ipynb`](0614a_solidos_semelhantes_e_troncos.ipynb) (o tronco de cone e o Pitágoras $g^2 = h^2 + (R - r)^2$)
- [`0615a_inscricao_e_circunscricao_esfera_e_poliedros.ipynb`](0615a_inscricao_e_circunscricao_esfera_e_poliedros.ipynb) (esferas inscrita e circunscrita; $r = \frac{3 \cdot \text{volume}}{S}$)
- [`0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb`](../04_geometria_plana/0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb) (semelhança de triângulos), [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (circunferências inscrita e circunscrita; $S = p \cdot r$) e [`0412a_circunferencia_e_circulo.ipynb`](../04_geometria_plana/0412a_circunferencia_e_circulo.ipynb) (quadriláteros circunscritíveis: somas dos lados opostos iguais)

**Isso será usado em:**
- [`0617a_superficies_e_solidos_de_revolucao.ipynb`](0617a_superficies_e_solidos_de_revolucao.ipynb) (a seção meridiana vira a "figura que gira")
- [`0618a_superficies_e_solidos_esfericos.ipynb`](0618a_superficies_e_solidos_esfericos.ipynb) (a técnica da seção meridiana aplicada às partes da esfera)
- 📌 Limite, Integral e Derivada (problemas de máximo: os desafios deste notebook)